# BioMed AI Nexus — Exploratory Data Analysis

Indian Liver Patient Dataset (ILPD). This notebook explores the data and
motivates the preprocessing + modelling choices used by `train.py`.

Run from the project root: `jupyter notebook notebooks/exploratory_analysis.ipynb`

In [ ]:
import sys, os
sys.path.append(os.path.abspath('..'))
import pandas as pd, numpy as np
import matplotlib.pyplot as plt
from utils.preprocessing import load_raw_dataset, encode_gender, dataset_overview
from config import FEATURE_ORDER

df = load_raw_dataset('../data/ilpd_raw.csv')
print(df.shape)
df.head()

## 1. Overview & missing values

In [ ]:
print(dataset_overview(df))
print('\nMissing values per column:')
print(df.isna().sum())
df.describe().T

## 2. Target balance
The dataset is imbalanced (~71% liver-disease cases), which motivates the balanced class weights used in training.

In [ ]:
df['Target'].map({1:'Disease',0:'Healthy'}).value_counts().plot.bar(
    color=['#ef4444','#10b981']); plt.title('Class balance'); plt.show()

## 3. Feature distributions by class

In [ ]:
num = [c for c in FEATURE_ORDER if c != 'Gender']
fig, axes = plt.subplots(3, 3, figsize=(13, 9))
for ax, col in zip(axes.ravel(), num):
    for t, c in [(1, '#ef4444'), (0, '#10b981')]:
        ax.hist(df[df['Target']==t][col].dropna(), bins=25, alpha=.6, color=c,
                label=('Disease' if t else 'Healthy'))
    ax.set_title(col, fontsize=9); ax.legend(fontsize=7)
plt.tight_layout(); plt.show()

## 4. Correlation matrix

In [ ]:
corr = encode_gender(df)[FEATURE_ORDER + ['Target']].astype(float).corr()
plt.figure(figsize=(8,7)); plt.imshow(corr, cmap='RdBu_r', vmin=-1, vmax=1)
plt.xticks(range(len(corr)), corr.columns, rotation=90, fontsize=7)
plt.yticks(range(len(corr)), corr.columns, fontsize=7)
plt.colorbar(); plt.title('Feature correlation'); plt.tight_layout(); plt.show()
# Note the strong Total/Direct Bilirubin and ALT/AST correlations.

## 5. Takeaways
- 4 missing values in *Albumin/Globulin Ratio* → median imputation.
- Features are on very different scales (Alk. Phos. in the hundreds vs A/G ~1) → standardisation.
- Class imbalance → balanced class weights + AUC-based model selection.
- Bilirubin and the aminotransferases are the most discriminative biomarkers — confirmed by the trained models' feature importance.

Train the models with: `python ../train.py`